# Module 6: LoRA & Domain Adaptation

> **Course: Training MiniMind LLM from Scratch**  |  *Google Colab NLP Course*

## 🎯 Learning Objectives
- Understand the **LoRA (Low-Rank Adaptation)** technique mathematically
- Apply LoRA to MiniMind and train with a fraction of the parameters
- Build your own **custom domain dataset** for personality / knowledge injection
- Merge LoRA weights back into the base model for efficient deployment

---

## 🧮 LoRA — The Math

Full fine-tuning updates every weight matrix $W \in \mathbb{R}^{d \times k}$.  
LoRA instead *freezes* $W$ and learns a **low-rank decomposition** of the update:

$$W' = W + \Delta W = W + B A$$

where $B \in \mathbb{R}^{d \times r}$ and $A \in \mathbb{R}^{r \times k}$ with rank $r \ll \min(d, k)$.

**Why this works:**
- Most weight updates during fine-tuning have **low intrinsic rank**
- With $r=8$ and $d=k=768$: only $2 \times 768 \times 8 = 12{,}288$ params per layer vs $768^2 = 589{,}824$
- **99%+ memory reduction** in trainable parameters!

**Scaling factor $\alpha$:** In practice LoRA scales the update by $\frac{\alpha}{r}$ to make  
learning-rate tuning independent of rank choice.

### Where LoRA is applied
LoRA is typically applied to the **attention projection matrices** ($Q$, $K$, $V$, $O$)  
and sometimes to the MLP layers — the high-dimensional linear transformations.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────
import subprocess, sys, os

def run(cmd): return subprocess.run(cmd, shell=True, check=False)

run("pip install -q torch transformers modelscope tqdm matplotlib")

if not os.path.exists('/content/minimind-colab'):
    run("git clone https://github.com/your-org/minimind-colab /content/minimind-colab")

sys.path.insert(0, '/content/minimind-colab')

run("nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader 2>/dev/null || echo 'No GPU'")

import torch
print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## 📖 Understanding the LoRA Implementation (`model/model_lora.py`)

Let's read the actual implementation to see exactly how LoRA is wired into MiniMind.

In [ ]:
with open('/content/minimind-colab/model/model_lora.py') as f:
    src = f.read()
print(src)


## ⚙️ LoRA Training Configuration

Notice the **higher learning rate** (`1e-4`) — with far fewer trainable parameters, LoRA can afford a larger LR without destabilising the frozen base weights.

In [ ]:
import argparse, os, torch

args = argparse.Namespace(
    save_dir='/content/minimind-colab/out',
    save_weight='full_sft',      # base model to load
    lora_name='lora_custom',     # name for saving LoRA delta weights
    data_path='/content/minimind-colab/dataset/sft_t2t_mini.jsonl',
    hidden_size=768, num_hidden_layers=8, use_moe=0,
    epochs=1, batch_size=8, learning_rate=1e-4,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    dtype='bfloat16', num_workers=2,
    accumulation_steps=1, grad_clip=1.0,
    log_interval=50, save_interval=500,
    max_seq_len=512,
)
os.makedirs(args.save_dir, exist_ok=True)
print("LoRA Configuration:")
for k, v in vars(args).items():
    print(f"  {k:<22} = {v}")


In [ ]:
import sys
sys.path.insert(0, '/content/minimind-colab')

from model.model_minimind import MiniMindConfig, MiniMindForCausalLM
from model.model_lora import apply_lora, save_lora, load_lora, merge_lora
from dataset.lm_dataset import SFTDataset
from trainer.trainer_utils import setup_seed, get_model_params, SkipBatchSampler, get_lr, init_model
from contextlib import nullcontext
from torch import optim
from torch.utils.data import DataLoader

setup_seed(42)
lm_config = MiniMindConfig(hidden_size=args.hidden_size,
                            num_hidden_layers=args.num_hidden_layers)

# Load the SFT-trained base model
model, tokenizer = init_model(lm_config, args.save_weight,
                               tokenizer_path='/content/minimind-colab/model',
                               save_dir=args.save_dir, device=args.device)

total_params = sum(p.numel() for p in model.parameters())
print(f"Base model total parameters: {total_params/1e6:.2f}M")

# Inject LoRA adapters
apply_lora(model)

# Only train LoRA parameters
lora_params  = [p for n, p in model.named_parameters() if 'lora_' in n]
trainable    = sum(p.numel() for p in lora_params)
frozen       = total_params - trainable

print(f"\nAfter apply_lora():")
print(f"  Trainable (LoRA) : {trainable/1e3:>8.1f} K  ({trainable/total_params*100:.3f}%)")
print(f"  Frozen (base)    : {frozen/1e6:>8.2f} M  ({frozen/total_params*100:.3f}%)")
print(f"  Memory savings   : {frozen/total_params*100:.1f}%  fewer gradients needed!")

# Verify that base weights are frozen
non_lora_grad = sum(p.numel() for n, p in model.named_parameters()
                    if 'lora_' not in n and p.requires_grad)
print(f"  Non-LoRA trainable params: {non_lora_grad}  (should be 0)")


## 📝 Create Your Custom Dataset

This is your chance to **teach MiniMind a new persona or domain knowledge**!  
Edit `custom_data` below with your own question-answer pairs.  
If you provide ≥ 10 examples, they will be used instead of the default SFT data.

In [ ]:
import json, os

# ✏️  EDIT THIS — add your own conversations!
custom_data = [
    {"conversations": [
        {"role": "user",      "content": "你是谁？"},
        {"role": "assistant", "content": "我是MiniMind，一个由NLP课程学生亲手训练的小型语言模型！"},
    ]},
    {"conversations": [
        {"role": "user",      "content": "你的特长是什么？"},
        {"role": "assistant", "content": "我擅长回答关于人工智能和机器学习的问题，我是用PyTorch从零开始训练的！"},
    ]},
    {"conversations": [
        {"role": "user",      "content": "你是用什么技术实现的？"},
        {"role": "assistant", "content": "我基于Transformer架构，使用了多头注意力机制和前馈神经网络，参数量约85M。"},
    ]},
    # ── Add more examples below this line ──────────────────────
    # {"conversations": [
    #     {"role": "user",      "content": "YOUR QUESTION"},
    #     {"role": "assistant", "content": "YOUR ANSWER"},
    # ]},
]

custom_path = '/content/minimind-colab/dataset/custom_lora.jsonl'
with open(custom_path, 'w', encoding='utf-8') as f:
    for item in custom_data:
        f.write(json.dumps(item, ensure_ascii=False) + '\n')

print(f"✅ Custom dataset saved: {len(custom_data)} samples → {custom_path}")
print(f"   Need ≥ 10 samples to use as primary training data (you have {len(custom_data)})")


In [ ]:
# Pick data source: custom if large enough, else download the mini SFT set
if len(custom_data) >= 10:
    data_path = custom_path
    print(f"📌 Using custom dataset ({len(custom_data)} samples)")
else:
    data_path = args.data_path
    print("📌 Custom dataset too small — using sft_t2t_mini.jsonl")
    if not os.path.exists(data_path):
        !modelscope download --model gongjy/minimind_dataset sft_t2t_mini.jsonl             --local_dir /content/minimind-colab/dataset

train_ds = SFTDataset(data_path, tokenizer, max_length=args.max_seq_len)

# Optimizer trains ONLY the LoRA params
optimizer     = optim.AdamW(lora_params, lr=args.learning_rate, weight_decay=0.01)
dtype_t       = torch.bfloat16 if args.dtype == 'bfloat16' else torch.float16
autocast_ctx  = (nullcontext() if 'cpu' in args.device
                 else torch.cuda.amp.autocast(dtype=dtype_t))
scaler        = torch.cuda.amp.GradScaler(enabled=(args.dtype == 'float16'))

print(f"Training dataset : {len(train_ds):,} samples")
print(f"Optimizer params : {sum(p.numel() for pg in optimizer.param_groups for p in pg['params'])/1e3:.1f}K")


## 🏋️ LoRA Training Loop

The loop is identical to SFT, except:
- `optimizer` only updates LoRA parameters
- Gradients do **not** flow through frozen base weights (greatly reduces VRAM)

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from IPython.display import clear_output
from tqdm.notebook import tqdm

loss_history, step_history = [], []

def lora_train(epochs=1):
    model.train()
    global_step = 0
    for epoch in range(epochs):
        indices      = torch.randperm(len(train_ds)).tolist()
        batch_sampler = SkipBatchSampler(indices, args.batch_size, 0)
        loader       = DataLoader(train_ds, batch_sampler=batch_sampler,
                                   num_workers=args.num_workers, pin_memory=True)
        iters  = len(loader)
        pbar   = tqdm(loader, desc=f"Epoch {epoch+1}/{epochs}")

        for step, (input_ids, labels) in enumerate(pbar, start=1):
            global_step += 1
            input_ids = input_ids.to(args.device)
            labels    = labels.to(args.device)

            lr = get_lr(epoch * iters + step, epochs * iters, args.learning_rate)
            for pg in optimizer.param_groups:
                pg['lr'] = lr

            with autocast_ctx:
                res  = model(input_ids, labels=labels)
                loss = (res.loss + res.aux_loss) / args.accumulation_steps

            scaler.scale(loss).backward()

            if step % args.accumulation_steps == 0:
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(lora_params, args.grad_clip)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)

            cur = loss.item() * args.accumulation_steps
            loss_history.append(cur)
            step_history.append(global_step)
            pbar.set_postfix({'loss': f'{cur:.4f}', 'lr': f'{lr:.2e}'})

            if global_step % args.log_interval == 0:
                clear_output(wait=True)
                fig, ax = plt.subplots(figsize=(12, 4))
                ax.plot(step_history, loss_history, 'b-', alpha=0.4,
                        linewidth=0.8, label='Loss')
                if len(loss_history) > 20:
                    w  = 20
                    sm = [sum(loss_history[max(0,i-w):i+1]) / min(i+1, w)
                          for i in range(len(loss_history))]
                    ax.plot(step_history, sm, 'r-', linewidth=2, label='Smoothed')
                ax.set_title(f'LoRA Training Loss  (step {global_step})')
                ax.set_xlabel('Step'); ax.set_ylabel('Loss')
                ax.legend(); ax.grid(True, alpha=0.3)
                plt.tight_layout(); plt.show()

lora_train(args.epochs)


In [ ]:
# Save LoRA delta weights only (very small file!)
lora_path = f'{args.save_dir}/{args.lora_name}_{lm_config.hidden_size}.pth'
save_lora(model, lora_path)
size_kb = os.path.getsize(lora_path) / 1e3
print(f"✅ LoRA weights saved: {lora_path}  ({size_kb:.1f} KB)")
print(f"   Compare to full model: {os.path.getsize(f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}.pth')/1e6:.1f} MB")


## 🧪 Test: Base Model + LoRA vs Fresh Reload

We test that the LoRA adapter is actually modifying behaviour, especially on your custom prompts.

In [ ]:
import torch

def generate(m, tok, prompt, device='cuda', max_new_tokens=120):
    m.eval()
    conv    = [{"role": "user", "content": prompt}]
    text    = tok.apply_chat_template(conv, tokenize=False, add_generation_prompt=True)
    inp     = tok(text, return_tensors="pt").to(device)
    with torch.no_grad():
        out = m.generate(inp["input_ids"], attention_mask=inp["attention_mask"],
                         max_new_tokens=max_new_tokens, do_sample=True,
                         temperature=0.7, pad_token_id=tok.pad_token_id,
                         eos_token_id=tok.eos_token_id)
    return tok.decode(out[0][inp["input_ids"].shape[1]:], skip_special_tokens=True)

# Reload base model WITHOUT LoRA for a fair comparison
from model.model_minimind import MiniMindForCausalLM, MiniMindConfig
base_model_clean = MiniMindForCausalLM(lm_config)
base_ckp         = f'{args.save_dir}/{args.save_weight}_{lm_config.hidden_size}.pth'
base_model_clean.load_state_dict(torch.load(base_ckp, map_location=args.device), strict=False)
base_model_clean = base_model_clean.half().eval().to(args.device)

test_prompts = ["你是谁？", "你是用什么技术实现的？", "介绍一下人工智能"]

for p in test_prompts:
    print(f"\n{'='*60}")
    print(f"❓ {p}")
    print(f"🔵 Base SFT : {generate(base_model_clean, tokenizer, p, args.device)[:200]}")
    print(f"🟢 + LoRA   : {generate(model, tokenizer, p, args.device)[:200]}")

del base_model_clean


## 🔀 Merging LoRA into Base Model

For deployment you want a **single weight file** — no runtime LoRA overhead.
`merge_lora(model)` absorbs $\Delta W = BA$ directly into $W$: $W \leftarrow W + BA$.

After merging, the model is equivalent to a fully fine-tuned model but built from the LoRA update.

In [ ]:
# Merge LoRA delta into base weights in-place
merge_lora(model)
print("✅ LoRA weights merged into base model.")

# Verify: LoRA params should now be zero (or absent)
lora_check = [(n, p.abs().max().item())
              for n, p in model.named_parameters() if 'lora_' in n]
if lora_check:
    print(f"LoRA param magnitudes after merge (should be ~0): "
          f"{[f'{v:.6f}' for _, v in lora_check[:3]]}")

# Save the merged model as a normal checkpoint
merged_path = f'{args.save_dir}/lora_merged_{lm_config.hidden_size}.pth'
torch.save({k: v.half().cpu() for k, v in model.state_dict().items()}, merged_path)
size_mb = os.path.getsize(merged_path) / 1e6
print(f"✅ Merged model saved: {merged_path}  ({size_mb:.1f} MB)")


In [ ]:
import gc, torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated()/1e9:.2f} GB allocated")


## 📝 Student Exercise

1. **Try different ranks**: Change the LoRA rank `r` inside `model/model_lora.py` (try `r=4`, `r=16`, `r=32`).  
   How does rank affect: (a) number of trainable params, (b) final loss, (c) quality of generated text?

2. **Target different layers**: LoRA is usually applied to Q/K/V/O projections.  
   Try applying it **only to Q and V** (as in the original LoRA paper). Does it still work?

3. **Load and re-apply**: Write code that starts from the merged model, loads the raw LoRA delta from `lora_custom_768.pth`, and re-applies it.  
   Why might you want to keep LoRA weights separate rather than merging?

## �� Discussion Questions

- LoRA freezes the base weights completely. What happens if the base model was trained on **very different data** from your fine-tuning set?
- Why is LoRA especially useful for **multi-tenant** LLM deployment (serving many users with different fine-tunes)?
- The original LoRA paper applies adapters to **attention** layers only. Newer work (e.g. DoRA, LoRA+) extends this. What are the trade-offs?
- How does LoRA compare to **Adapter tuning** and **Prefix tuning**?
